<a href="https://colab.research.google.com/github/Kanakbaghel/Vaani-Noise_Event_Detection/blob/main/notebooks/detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets huggingface_hub

In [2]:
!pip install datasets

In [3]:
from datasets import load_dataset
from google.colab import userdata

# Retrieve your Hugging Face token from Colab secrets.
# IMPORTANT: Ensure your Hugging Face token is stored in Colab secrets with the name 'HF_TOKEN'.
# Click on the 'Secrets' icon on the left sidebar to add/verify your secret.
hf_token = userdata.get('HF_TOKEN')

# Load the dataset passing your token explicitly in streaming mode
ds = load_dataset("ARTPARK-IISc/Vaani-Noise-Event-Dataset", token=hf_token)

print("Raw dataset loaded.")

# Get the 'train' split as an IterableDataset.
# This variable will be used for iteration to create the DataFrame.
ds_train_iterable = ds['train']

print("Train split (IterableDataset) is ready for iteration.")

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

Raw dataset loaded.
Train split (IterableDataset) is ready for iteration.


In [4]:
# Display the dataset features to understand available columns
print(ds['train'].features)

{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'imageFileName': Value('string'), 'state': Value('string'), 'district': Value('string'), 'duration': Value('float64'), 'language': Value('string'), 'annotationQuality': Value('string'), 'isTranscriptionAvailable': Value('bool'), 'transcript': Value('string'), 'NoiseCategory': List(Value('string')), 'NoiseSubCategoryTimeStamp': List({'category': Value('string'), 'tag': Value('string'), 'start': Value('string'), 'end': Value('string')})}


In [5]:
# Access the first sample in the dataset
sample = ds['train'][0]

print("State:", sample['state'])
print("Language:", sample['language'])
print("Transcript:", sample['transcript'])
print("Noise Categories:", sample['NoiseCategory'])
print("Timestamps:", sample['NoiseSubCategoryTimeStamp'])

State: AndhraPradesh
Language: Telugu
Transcript: <static noise> ఈ ఏరును చూసి ఆనందించడానికి కూర్చోడానికి ఒక [breathing] సిమెంట్ {Cement} కూర్చి కూడా చేసి ఉండడం [breathing] మనం కనిపిస్తుంది. </static noise>
Noise Categories: ['human_non_speech']
Timestamps: []


In [6]:
import IPython.display as ipd

# When the dataset is loaded with `streaming=True`, ds['train'][0] returns IterableColumn objects.
# To get a concrete sample from an IterableDataset, we need to iterate over it.
# Let's get the first materialized sample:
sample = next(iter(ds['train']))

# Extract audio array and sampling rate
audio_data = sample['audio']['array']
sampling_rate = sample['audio']['sampling_rate']

# Display player
ipd.Audio(audio_data, rate=sampling_rate)

In [7]:
# Parse noise timestamps for a sample
for event in sample['NoiseSubCategoryTimeStamp']:
    category = event.get('category')
    start_time = event.get('start')
    end_time = event.get('end')
    print(f"Noise Type: {category} | Onset: {start_time}s | Offset: {end_time}s")

In [8]:
# To avoid memory issues with streaming datasets, we will inspect a limited number of samples.
# Let's collect a few samples to check for missing values and annotation quality.
num_samples_to_check = 100

samples_data = []
annotation_qualities = []

# Iterate over a limited number of samples from the streaming dataset
for i, sample in enumerate(ds['train']):
    if i >= num_samples_to_check:
        break
    samples_data.append(sample)
    annotation_qualities.append(sample.get('annotationQuality'))

# Convert collected samples to a pandas DataFrame for easier inspection
import pandas as pd
samples_df = pd.DataFrame(samples_data)

print(f"--- Data verification for the first {num_samples_to_check} samples ---")

print("\nMissing values per column in collected samples:")
print(samples_df.isnull().sum())

print("\nUnique values and counts for 'annotationQuality' in collected samples:")
if annotation_qualities:
    quality_series = pd.Series(annotation_qualities)
    print(quality_series.value_counts(dropna=False)) # dropna=False to count None/NaNs
else:
    print("No annotation quality data collected.")

print("\nData types of columns in collected samples:")
print(samples_df.info())

--- Data verification for the first 100 samples ---

Missing values per column in collected samples:
audio                        0
imageFileName                0
state                        0
district                     0
duration                     0
language                     0
annotationQuality            0
isTranscriptionAvailable     0
transcript                   0
NoiseCategory                0
NoiseSubCategoryTimeStamp    0
dtype: int64

Unique values and counts for 'annotationQuality' in collected samples:
unverified_timestamps    52
verified_timestamps      31
no_timestamps            17
Name: count, dtype: int64

Data types of columns in collected samples:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   audio                      100 non-null    object 
 1   imageFileName              100 non-null  